# VSense aktivite sınıflandırma baseline'ı

Bu notebook Hafta 6 için pencereleme, özellik çıkarımı, oturum bazlı veri ayrımı, kNN/SVM eğitimi ve dürüst değerlendirme akışını belgeler. Komşu pencerelerin train ve test kümelerine sızmasını önlemek için ayrım oturum tekrarı üzerinden yapılır.

## 1. Pencereleri ve özellik tablosunu üretme

Varsayılan ayar iki saniyelik pencere, bir saniyelik kayma, ilk/son 10 saniyeyi kırpma ve 500 ms maksimum pencere içi boşluktur.

In [ ]:
# Repo kökünden çalıştırın:
# !PYTHONPATH=server server/.venv/bin/python server/ml/build_features.py

In [ ]:
from pathlib import Path
import pandas as pd

features_path = Path('../dataset-v1/processed/features.parquet')
features = pd.read_parquet(features_path)
features.groupby(['label', 'session_id']).size().rename('windows')

## 2. Dürüst veri ayrımı

- Repeat 1: train
- Repeat 2: validation
- Repeat 3: test

Test kümesi model veya hiperparametre seçmek için kullanılmaz.

In [ ]:
# Repo kökünden çalıştırın:
# !MPLCONFIGDIR=/tmp/vsense-matplotlib PYTHONPATH=server server/.venv/bin/python server/ml/train_baselines.py

In [ ]:
import json

metrics_path = Path('../dataset-v1/models/baseline_v1/metrics.json')
metrics = json.loads(metrics_path.read_text())
pd.DataFrame({
    name: {
        'validation_macro_f1': result['validation']['macro_f1'],
        'test_accuracy': result['test']['accuracy'],
        'test_macro_f1': result['test']['macro_f1'],
    }
    for name, result in metrics['models'].items()
}).T

In [ ]:
from IPython.display import Image, display
display(Image('../dataset-v1/models/baseline_v1/confusion_matrix_knn.png'))
display(Image('../dataset-v1/models/baseline_v1/confusion_matrix_svm.png'))

## 3. Dürüstlük bölümü

İlk baseline testte yaklaşık 0.27 macro-F1 vermiştir. Walking ve desk_work kısmen ayrılırken empty_room, sitting ve standing genellenememiştir. Bu sonuç saklanmamalıdır: CSI oturum dağılımı değişmekte ve katılımcılar sınıflar arasında dengeli değildir. Mevcut test oturum-bağımsızdır fakat tam kişi/gün bağımsız değildir. Hafta 6 DoD'si için bütün sınıfları kapsayan farklı gün ve kişi holdout kaydı ayrıca alınmalıdır.